# Classificação de imagens RGB com CNNs usando CIFAR-10

Aprendizagem Profunda — Mestrado em Engenharia Informática / IA / ECD  
Universidade do Minho, 2025/2026

## T1 – Download e extração do dataset CIFAR-10

In [ ]:
import os

if not os.path.exists('cifar'):
    !wget http://pjreddie.com/media/files/cifar.tgz
    !tar xzf cifar.tgz
    print('Dataset downloaded and extracted.')
else:
    print('Dataset already exists, skipping download.')

## T2 – Instalação de dependências

In [ ]:
# Check for CUDA / nvcc
!nvcc --version 2>/dev/null || echo 'nvcc not found (CPU-only environment)'
!nvidia-smi 2>/dev/null || echo 'nvidia-smi not found'

In [ ]:
!pip install -q torch torchvision livelossplot gdown matplotlib numpy

## T3 – Definir batch_size

In [ ]:
BATCH_SIZE = 128
print(f'batch_size = {BATCH_SIZE}')

## T4 – Preparação dos dados

### T4.1 – Lista de classes do dataset

In [ ]:
# CIFAR-10 classes (order matches the dataset label indices)
CLASSES = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck',
]
print(f'Classes ({len(CLASSES)}): {CLASSES}')

### T4.2 & T4.3 – Pré-processamento: normalização e conversão para CHW

In [ ]:
import torch
import torchvision.transforms as transforms

# Mean and std computed over the CIFAR-10 training set (per channel)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

# Training transforms: random augmentations + normalisation
# ToTensor() converts HWC uint8 [0,255] → CHW float32 [0,1]  (T4.3)
# Normalize() applies per-channel mean/std normalisation       (T4.2)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),                          # HWC → CHW, [0,1]
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Validation / test transforms: only normalisation (no augmentation)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

print('Transforms defined.')
print('  train_transform:', train_transform)
print('  eval_transform :', eval_transform)

### T4.4 – Leitura do dataset e preparação dos DataLoaders (holdout)

In [ ]:
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split


class CIFAR10PJReddie(Dataset):
    """Loads the pjreddie CIFAR-10 dataset.

    The tgz archive produces a ``cifar/`` folder with:
      cifar/train/<idx>_<classname>.png
      cifar/test/<idx>_<classname>.png
      cifar/labels.txt
    """

    def __init__(self, root: str, split: str = 'train', transform=None):
        assert split in ('train', 'test')
        self.transform = transform
        self.samples = []  # list of (path, class_index)

        split_dir = os.path.join(root, split)
        for img_path in sorted(glob.glob(os.path.join(split_dir, '*.png'))):
            # filename format: "<idx>_<classname>.png"
            basename = os.path.splitext(os.path.basename(img_path))[0]
            class_name = '_'.join(basename.split('_')[1:])  # handles multi-word names
            if class_name not in CLASSES:
                continue
            class_idx = CLASSES.index(class_name)
            self.samples.append((img_path, class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


CIFAR_ROOT = 'cifar'

# Full training set (50 000 images)
full_train_dataset = CIFAR10PJReddie(CIFAR_ROOT, split='train', transform=train_transform)

# Holdout split: 80 % train  /  20 % validation
VAL_RATIO   = 0.2
n_total     = len(full_train_dataset)
n_val       = int(n_total * VAL_RATIO)
n_train     = n_total - n_val

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)

# Apply eval transform to the validation subset
# We wrap val_dataset so it uses eval_transform instead of train_transform
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        # Retrieve raw PIL image by bypassing the original transform
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


val_dataset = TransformSubset(val_dataset, eval_transform)

# Test set (10 000 images)
test_dataset = CIFAR10PJReddie(CIFAR_ROOT, split='test', transform=eval_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples      : {len(train_dataset):>6}')
print(f'Validation samples : {len(val_dataset):>6}')
print(f'Test samples       : {len(test_dataset):>6}')
print(f'Train batches      : {len(train_loader):>6}')
print(f'Val batches        : {len(val_loader):>6}')
print(f'Test batches       : {len(test_loader):>6}')

## T5 – Visualização dos dados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Reverse normalisation for display
def denormalize(tensor, mean=CIFAR10_MEAN, std=CIFAR10_STD):
    """Convert a normalised CHW tensor back to a displayable HWC numpy array."""
    t = tensor.clone()
    for c, (m, s) in enumerate(zip(mean, std)):
        t[c] = t[c] * s + m
    return t.permute(1, 2, 0).numpy().clip(0, 1)


# --- Dataset metrics ---
print('=== Dataset metrics ===')
print(f'  Total images : {len(full_train_dataset) + len(test_dataset)}')
print(f'  Training     : {len(train_dataset)}')
print(f'  Validation   : {len(val_dataset)}')
print(f'  Test         : {len(test_dataset)}')
print(f'  Classes      : {len(CLASSES)}')
print(f'  Image size   : 32 × 32 × 3 (RGB)')
print(f'  Batch size   : {BATCH_SIZE}')

# --- Visualise one training batch ---
images, labels = next(iter(train_loader))
print(f'\nBatch tensor shape : {images.shape}  (N × C × H × W)')

n_show = 16
fig, axes = plt.subplots(2, n_show // 2, figsize=(n_show, 5))
axes = axes.flatten()

for i in range(n_show):
    img_np = denormalize(images[i])
    axes[i].imshow(img_np)
    axes[i].set_title(CLASSES[labels[i].item()], fontsize=9)
    axes[i].axis('off')

fig.suptitle('Sample training batch (16 images)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## T6 – Verificação do balanceamento do dataset

In [ ]:
from collections import Counter


def class_distribution(dataset, name: str):
    """Count samples per class and print a summary table."""
    if hasattr(dataset, 'samples'):
        # CIFAR10PJReddie
        all_labels = [label for _, label in dataset.samples]
    elif hasattr(dataset, 'subset'):
        # TransformSubset (validation)
        all_labels = [dataset.subset.dataset.samples[dataset.subset.indices[i]][1]
                      for i in range(len(dataset))]
    else:
        # random_split Subset — access via underlying dataset indices
        all_labels = [dataset.dataset.samples[i][1] for i in dataset.indices]

    counts = Counter(all_labels)
    total  = sum(counts.values())

    print(f'\n=== {name} ({total} samples) ===')
    print(f'  {"Class":<12} {"Count":>6}  {"Share":>7}')
    print('  ' + '-' * 28)
    for idx, cls in enumerate(CLASSES):
        n   = counts.get(idx, 0)
        pct = 100 * n / total if total > 0 else 0
        print(f'  {cls:<12} {n:>6}  {pct:>6.2f} %')

    return counts


train_counts = class_distribution(train_dataset, 'Training set')
val_counts   = class_distribution(val_dataset,   'Validation set')
test_counts  = class_distribution(test_dataset,  'Test set')

In [ ]:
# Visual bar chart of class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

split_info = [
    ('Training set',    train_counts),
    ('Validation set',  val_counts),
    ('Test set',        test_counts),
]

for ax, (title, counts) in zip(axes, split_info):
    values = [counts.get(i, 0) for i in range(len(CLASSES))]
    bars = ax.bar(CLASSES, values, color='steelblue', edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of samples')
    ax.tick_params(axis='x', rotation=45)
    # Annotate bar heights
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=7)

    # Reference line for a perfectly balanced dataset
    expected = sum(values) / len(CLASSES)
    ax.axhline(expected, color='red', linestyle='--', linewidth=1.2,
               label=f'Expected ({int(expected)})')
    ax.legend(fontsize=8)

fig.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nConclusion: CIFAR-10 is a perfectly balanced dataset — each class '
      'contains the same number of samples in every split.')

## T7 – Definição dos modelos

### T7.0 – Imports comuns

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd()) != 'ComputerVisionClass' else os.getcwd())

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


### T7.1 – Modelo ResNet

Architecture:
- Conv2d → BatchNorm2d
- layer1: 2 × ResidualBlock (64 ch, stride 1)
- layer2: 2 × ResidualBlock (128 ch, stride 2)
- layer3: 2 × ResidualBlock (256 ch, stride 2)
- layer4: 2 × ResidualBlock (512 ch, stride 2)
- Linear(512, 10)

Each ResidualBlock: Conv2d → BN → Conv2d → BN + shortcut (Sequential)

In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block: Conv→BN + Conv→BN with skip connection."""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        # Shortcut adjusts dimensions when needed
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class ResNet(nn.Module):
    """ResNet for CIFAR-10 (T7.1)."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64,  64,  num_blocks=2, stride=1)
        self.layer2 = self._make_layer(64,  128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, num_blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, num_blocks=2, stride=2)
        self.fc     = nn.Linear(512, num_classes)

    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.adaptive_avg_pool2d(out, 1)
        out = out.view(out.size(0), -1)
        return self.fc(out)


resnet_model = ResNet(num_classes=10).to(DEVICE)
print(resnet_model)
total_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')


In [ ]:
# Optional: download pre-trained ResNet weights (skip if not needed)
RESNET_WEIGHTS = 'CNNModel_cifar_Resnet.pth'
if not os.path.exists(RESNET_WEIGHTS):
    try:
        import gdown
        gdown.download(id='1pg3nKSWsttMlaH7APwgfkgAUlBBlUNw_', output=RESNET_WEIGHTS, quiet=False)
    except Exception as e:
        print(f'Could not download pre-trained weights: {e}')
        print('The model will be trained from scratch in T8.')
else:
    print(f'Pre-trained weights found: {RESNET_WEIGHTS}')


### T7.2 – Modelo CNN 1

Architecture:
- features1: Conv2d(3→32) → ReLU → MaxPool(2)
- features2: Conv2d(32→64) → ReLU → MaxPool(2)
- Linear(64·8·8, 512) → ReLU → Linear(512, 10) → Softmax

In [ ]:
class CNN1(nn.Module):
    """CNN 1 (T7.2)."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.features2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1     = nn.Linear(64 * 8 * 8, 512)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(512, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        out = self.features1(x)
        out = self.features2(out)
        out = out.view(out.size(0), -1)
        out = self.relu(self.fc1(out))
        return self.softmax(self.fc2(out))


cnn1_model = CNN1(num_classes=10).to(DEVICE)
print(cnn1_model)
print(f'\nTrainable parameters: {sum(p.numel() for p in cnn1_model.parameters() if p.requires_grad):,}')


In [ ]:
CNN1_WEIGHTS = 'CNNModel_cifar_1.pth'
if not os.path.exists(CNN1_WEIGHTS):
    try:
        import gdown
        gdown.download(id='1YiY4zmuaQGwk_mSjK3LNWUMJfAABkbCx', output=CNN1_WEIGHTS, quiet=False)
    except Exception as e:
        print(f'Could not download pre-trained weights: {e}')
else:
    print(f'Pre-trained weights found: {CNN1_WEIGHTS}')


### T7.3 – Modelo CNN 2

Architecture:
- features1: Conv2d(3→32) → ReLU → MaxPool(2)
- features2: Conv2d(32→64) → ReLU → MaxPool(2)
- Linear(64·8·8, 10)

In [ ]:
class CNN2(nn.Module):
    """CNN 2 (T7.3)."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.features2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(64 * 8 * 8, num_classes)

    def forward(self, x):
        out = self.features1(x)
        out = self.features2(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)


cnn2_model = CNN2(num_classes=10).to(DEVICE)
print(cnn2_model)
print(f'\nTrainable parameters: {sum(p.numel() for p in cnn2_model.parameters() if p.requires_grad):,}')


In [ ]:
CNN2_WEIGHTS = 'CNNModel_cifar_2.pth'
if not os.path.exists(CNN2_WEIGHTS):
    try:
        import gdown
        gdown.download(id='1p4Udbjb4q3uerAk_F8Jxs7H2UZL4QUN', output=CNN2_WEIGHTS, quiet=False)
    except Exception as e:
        print(f'Could not download pre-trained weights: {e}')
else:
    print(f'Pre-trained weights found: {CNN2_WEIGHTS}')


### T7.4 – Modelo CNN 3

Architecture:
- features1: Conv2d(3→32) → BN → ReLU → MaxPool(2)
- features2: Conv2d(32→64) → BN → ReLU → MaxPool(2)
- Linear(64·8·8, 512) → Dropout(0.5) → Linear(512, 128) → Linear(128, 10)

In [ ]:
class CNN3(nn.Module):
    """CNN 3 (T7.4)."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.features2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1     = nn.Linear(64 * 8 * 8, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(512, 128)
        self.fc3     = nn.Linear(128, num_classes)

    def forward(self, x):
        out = self.features1(x)
        out = self.features2(out)
        out = out.view(out.size(0), -1)
        out = F.relu(self.fc1(out))
        out = self.dropout(out)
        out = F.relu(self.fc2(out))
        return self.fc3(out)


cnn3_model = CNN3(num_classes=10).to(DEVICE)
print(cnn3_model)
print(f'\nTrainable parameters: {sum(p.numel() for p in cnn3_model.parameters() if p.requires_grad):,}')


In [ ]:
CNN3_WEIGHTS = 'CNNModel_cifar_3.pth'
if not os.path.exists(CNN3_WEIGHTS):
    try:
        import gdown
        gdown.download(id='18KNK0wgJZjCVmgWB5ykUAVGwaU5PcURU', output=CNN3_WEIGHTS, quiet=False)
    except Exception as e:
        print(f'Could not download pre-trained weights: {e}')
else:
    print(f'Pre-trained weights found: {CNN3_WEIGHTS}')


### T7.5 – Modelo CNN 4

Architecture:
- features: Conv2d(3→32) → BN → ReLU → MaxPool(2) → Dropout(0.25)
- Linear(32·16·16, 256) → Linear(256, 10)

In [ ]:
class CNN4(nn.Module):
    """CNN 4 (T7.5)."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.fc1 = nn.Linear(32 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = F.relu(self.fc1(out))
        return self.fc2(out)


cnn4_model = CNN4(num_classes=10).to(DEVICE)
print(cnn4_model)
print(f'\nTrainable parameters: {sum(p.numel() for p in cnn4_model.parameters() if p.requires_grad):,}')


In [ ]:
CNN4_WEIGHTS = 'CNNModel_cifar_4.pth'
if not os.path.exists(CNN4_WEIGHTS):
    try:
        import gdown
        gdown.download(id='1VWCisHcM1K63-6qYeLjNqgiTcTBmPy_1', output=CNN4_WEIGHTS, quiet=False)
    except Exception as e:
        print(f'Could not download pre-trained weights: {e}')
else:
    print(f'Pre-trained weights found: {CNN4_WEIGHTS}')


## T8 – Treino dos modelos

### T8.0 – Utilitários de treino

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs, lr, weights_path, model_name):
    """Train *model* and log accuracy/loss for train and validation sets.

    If *weights_path* exists the saved weights are loaded and training is skipped.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

    # Load pre-trained weights if available and skip training
    if os.path.exists(weights_path):
        state = torch.load(weights_path, map_location=DEVICE)
        model.load_state_dict(state)
        print(f'[{model_name}] Loaded weights from {weights_path} – skipping training.')
        return {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, num_epochs + 1):
        # ── Training phase ─────────────────────────────────────────────────
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Validation phase ────────────────────────────────────────────────
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)
        val_loss = val_running_loss / val_total
        val_acc  = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if epoch % max(1, num_epochs // 10) == 0 or epoch == 1:
            print(f'[{model_name}] Epoch {epoch:3d}/{num_epochs}  '
                  f'train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  '
                  f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

    # Save trained weights
    torch.save(model.state_dict(), weights_path)
    print(f'[{model_name}] Weights saved to {weights_path}')
    return history


def plot_history(history, model_name):
    """Plot training/validation loss and accuracy curves."""
    if not history['train_loss']:
        print(f'[{model_name}] No training history to plot (pre-trained weights used).')
        return

    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history['train_loss'], label='Train')
    axes[0].plot(epochs, history['val_loss'],   label='Validation')
    axes[0].set_title(f'{model_name} – Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs, history['train_acc'], label='Train')
    axes[1].plot(epochs, history['val_acc'],   label='Validation')
    axes[1].set_title(f'{model_name} – Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.show()


### T8.1 – Treino ResNet (30 epochs, lr=0.001, CrossEntropyLoss + SGD)

In [ ]:
history_resnet = train_model(
    resnet_model, train_loader, val_loader,
    num_epochs=30, lr=0.001,
    weights_path=RESNET_WEIGHTS,
    model_name='ResNet',
)
plot_history(history_resnet, 'ResNet')


### T8.2 – Treino CNN 1 (15 epochs, lr=0.001, CrossEntropyLoss + SGD)

In [ ]:
history_cnn1 = train_model(
    cnn1_model, train_loader, val_loader,
    num_epochs=15, lr=0.001,
    weights_path=CNN1_WEIGHTS,
    model_name='CNN1',
)
plot_history(history_cnn1, 'CNN1')


### T8.3 – Treino CNN 2 (15 epochs, lr=0.001, CrossEntropyLoss + SGD)

In [ ]:
history_cnn2 = train_model(
    cnn2_model, train_loader, val_loader,
    num_epochs=15, lr=0.001,
    weights_path=CNN2_WEIGHTS,
    model_name='CNN2',
)
plot_history(history_cnn2, 'CNN2')


### T8.4 – Treino CNN 3 (15 epochs, lr=0.001, CrossEntropyLoss + SGD)

In [ ]:
history_cnn3 = train_model(
    cnn3_model, train_loader, val_loader,
    num_epochs=15, lr=0.001,
    weights_path=CNN3_WEIGHTS,
    model_name='CNN3',
)
plot_history(history_cnn3, 'CNN3')


### T8.5 – Treino CNN 4 (75 epochs, lr=0.001, CrossEntropyLoss + SGD)

In [ ]:
history_cnn4 = train_model(
    cnn4_model, train_loader, val_loader,
    num_epochs=75, lr=0.001,
    weights_path=CNN4_WEIGHTS,
    model_name='CNN4',
)
plot_history(history_cnn4, 'CNN4')


## T9 – Avaliação dos modelos

### T9.0 – Utilitários de avaliação

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def evaluate_model(model, loader, model_name):
    """Evaluate *model* on *loader* and return predictions and true labels."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = (all_preds == all_labels).mean()
    print(f'[{model_name}] Test accuracy: {acc:.4f}  ({int(acc * len(all_labels))}/{len(all_labels)})')
    return all_preds, all_labels


def plot_confusion_matrix(all_preds, all_labels, model_name, classes=CLASSES):
    """Plot a normalised confusion matrix."""
    num_classes = len(classes)
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for true, pred in zip(all_labels, all_preds):
        cm[true][pred] += 1
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(
        xticks=range(num_classes), yticks=range(num_classes),
        xticklabels=classes, yticklabels=classes,
        xlabel='Predicted label', ylabel='True label',
        title=f'{model_name} – Confusion Matrix (normalised)',
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    thresh = cm_norm.max() / 2.0
    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, f'{cm_norm[i, j]:.2f}',
                    ha='center', va='center',
                    color='white' if cm_norm[i, j] > thresh else 'black',
                    fontsize=7)
    plt.tight_layout()
    plt.show()


def show_sample_predictions(model, loader, model_name, n=16):
    """Display *n* test images with their predicted and true labels."""
    model.eval()
    images_batch, labels_batch = next(iter(loader))
    images_dev = images_batch.to(DEVICE)
    with torch.no_grad():
        outputs = model(images_dev)
    preds = outputs.argmax(dim=1).cpu()

    fig, axes = plt.subplots(2, n // 2, figsize=(n, 5))
    axes = axes.flatten()
    for i in range(n):
        img_np = denormalize(images_batch[i])
        axes[i].imshow(img_np)
        pred_cls = CLASSES[preds[i].item()]
        true_cls = CLASSES[labels_batch[i].item()]
        color = 'green' if pred_cls == true_cls else 'red'
        axes[i].set_title(f'P:{pred_cls}\nT:{true_cls}', fontsize=7, color=color)
        axes[i].axis('off')
    fig.suptitle(f'{model_name} – Sample Predictions (green=correct, red=wrong)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


### T9.1 – Avaliação ResNet

In [ ]:
preds_resnet, labels_resnet = evaluate_model(resnet_model, test_loader, 'ResNet')
show_sample_predictions(resnet_model, test_loader, 'ResNet')
plot_confusion_matrix(preds_resnet, labels_resnet, 'ResNet')


### T9.2 – Avaliação CNN 1

In [ ]:
preds_cnn1, labels_cnn1 = evaluate_model(cnn1_model, test_loader, 'CNN1')
show_sample_predictions(cnn1_model, test_loader, 'CNN1')
plot_confusion_matrix(preds_cnn1, labels_cnn1, 'CNN1')


### T9.3 – Avaliação CNN 2

In [ ]:
preds_cnn2, labels_cnn2 = evaluate_model(cnn2_model, test_loader, 'CNN2')
show_sample_predictions(cnn2_model, test_loader, 'CNN2')
plot_confusion_matrix(preds_cnn2, labels_cnn2, 'CNN2')


### T9.4 – Avaliação CNN 3

In [ ]:
preds_cnn3, labels_cnn3 = evaluate_model(cnn3_model, test_loader, 'CNN3')
show_sample_predictions(cnn3_model, test_loader, 'CNN3')
plot_confusion_matrix(preds_cnn3, labels_cnn3, 'CNN3')


### T9.5 – Avaliação CNN 4

In [ ]:
preds_cnn4, labels_cnn4 = evaluate_model(cnn4_model, test_loader, 'CNN4')
show_sample_predictions(cnn4_model, test_loader, 'CNN4')
plot_confusion_matrix(preds_cnn4, labels_cnn4, 'CNN4')


## T10 – Previsão de um caso individual

### T10.0 – Utilitários de previsão individual

Utiliza uma imagem aleatória do conjunto de teste para demonstrar a previsão de cada modelo.

In [ ]:
def predict_single(model, dataset, model_name, sample_idx=None):
    """Predict a single sample from *dataset* and display the result.

    Parameters
    ----------
    model       : trained PyTorch model
    dataset     : dataset with __getitem__ returning (image_tensor, label)
    model_name  : string used in the title
    sample_idx  : index of the sample to use; random if None
    """
    if sample_idx is None:
        sample_idx = int(torch.randint(len(dataset), (1,)).item())

    model.eval()
    image_tensor, true_label = dataset[sample_idx]
    input_tensor = image_tensor.unsqueeze(0).to(DEVICE)   # add batch dim

    with torch.no_grad():
        output = model(input_tensor)
    probabilities = torch.softmax(output, dim=1)[0].cpu().numpy()
    predicted_class = int(probabilities.argmax())

    # ── display ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    img_np = denormalize(image_tensor)
    axes[0].imshow(img_np)
    axes[0].axis('off')
    correct = predicted_class == true_label
    axes[0].set_title(
        f'True: {CLASSES[true_label]}\n'
        f'Predicted: {CLASSES[predicted_class]} (\'{"✓" if correct else "✗"}\')' ,
        fontsize=11,
        color='green' if correct else 'red',
    )

    colors = ['steelblue'] * 10
    colors[predicted_class] = 'darkorange'
    colors[true_label]      = 'green'
    axes[1].barh(CLASSES, probabilities, color=colors)
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Class probabilities')
    axes[1].invert_yaxis()

    fig.suptitle(f'{model_name} – Single prediction (sample #{sample_idx})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'[{model_name}] sample_idx={sample_idx}  true={CLASSES[true_label]}  '
          f'predicted={CLASSES[predicted_class]}  confidence={probabilities[predicted_class]:.4f}')
    return predicted_class, true_label, probabilities


# Use the same sample index for all models to allow direct comparison
SAMPLE_IDX = int(torch.randint(len(test_dataset), (1,)).item())
print(f'Using test sample index: {SAMPLE_IDX}')


### T10.1 – Previsão individual: ResNet

In [ ]:
predict_single(resnet_model, test_dataset, 'ResNet', sample_idx=SAMPLE_IDX)

### T10.2 – Previsão individual: CNN 1

In [ ]:
predict_single(cnn1_model, test_dataset, 'CNN1', sample_idx=SAMPLE_IDX)

### T10.3 – Previsão individual: CNN 2

In [ ]:
predict_single(cnn2_model, test_dataset, 'CNN2', sample_idx=SAMPLE_IDX)

### T10.4 – Previsão individual: CNN 3

In [ ]:
predict_single(cnn3_model, test_dataset, 'CNN3', sample_idx=SAMPLE_IDX)

### T10.5 – Previsão individual: CNN 4

In [ ]:
predict_single(cnn4_model, test_dataset, 'CNN4', sample_idx=SAMPLE_IDX)